<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">شکل درست، محاسبهٔ غلط</h1>
<p style="text-align:right">درس 67 از 92 · از پیام خطا تا علت اصلی · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">60-bug-clinic</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-05/60-bug-clinic.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">محور کلاس در <bdi dir="ltr">Cross-Entropy</bdi> و ترتیب واقعی خانه‌ها در تقسیم <bdi dir="ltr">Head</bdi>ها را آزمایش کنید.</p><p style="text-align:right">پیش‌نیاز: <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">reshape</code>، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">transpose</code> و معنای محورهای <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,T,V)</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,T,C)</code>.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۶۰–۱۱۰ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">وقتی <bdi dir="ltr">T</bdi> و <bdi dir="ltr">V</bdi> برابر باشند، چرا یک خطای محور ممکن است بدون پیام خطا اجرا شود؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import torch
from torch.nn import functional as F
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
torch.manual_seed(7)
model = MiniGPT(ModelConfig(5,4,8,2,1,0.0))
x,y = torch.tensor([[1,2,3],[2,3,4]]),torch.tensor([[2,3,4],[3,4,1]])
logits,real_loss = model(x,y)
print('actual logits:',tuple(logits.shape),'actual targets:',tuple(y.shape))
try:
    F.cross_entropy(logits,y)
except RuntimeError as error:
    print('expected wrong-axis failure:',error)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">last_axis_loss(logits, targets)</code> برای <bdi dir="ltr">Logits</bdi> با شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,T,V)</code> و <bdi dir="ltr">Target</bdi> با شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,T)</code>، میانگین <bdi dir="ltr">Cross-Entropy</bdi> روی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">B*T</code> هدف را برگرداند. تست‌ها فقط <bdi dir="ltr">Shape</bdi> را بررسی نمی‌کنند.</p>
</div>

In [ ]:
def last_axis_loss(logits, targets):
    # TODO: محور Vocabulary باید محور کلاس باشد
    return None

In [ ]:
def test_exercise():
    result = last_axis_loss(logits,y)
    if result is None:
        return False
    torch.testing.assert_close(result,real_loss)
    for shape in ((2,3,3),(1,2,7)):
        scores = torch.arange(shape[0]*shape[1]*shape[2],dtype=torch.float32).reshape(shape)/7
        targets = torch.zeros(shape[:2],dtype=torch.long)
        expected = -scores.log_softmax(-1).gather(-1,targets[...,None]).mean()
        torch.testing.assert_close(last_axis_loss(scores,targets),expected)
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: last_axis_loss')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط برابر یا نابرابر بودن <bdi dir="ltr">T</bdi> و <bdi dir="ltr">V</bdi> را عوض کنید؛ اجرای بدون استثنا را با برابری مقدار اشتباه نگیرید.</p>
</div>

In [ ]:
scores = torch.arange(18,dtype=torch.float32).reshape(2,3,3)/4
targets = torch.tensor([[0,1,2],[2,0,1]])
wrong = F.cross_entropy(scores,targets)
correct = -scores.log_softmax(-1).gather(-1,targets[...,None]).mean()
print('same shape accepted; wrong:',wrong.item(),'correct:',correct.item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">reshape</code> مستقیم به <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,H,T,D)</code>، ترتیب خانه‌ها را مثل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">transpose</code> جابه‌جا نمی‌کند. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">split_heads(x,heads)</code> ورودی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,T,C)</code> را به <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,H,T,C/H)</code> با حفظ معنی خانه‌ها تبدیل کند.</p>
</div>

In [ ]:
values = torch.arange(2*3*8).reshape(2,3,8)
wrong = values.reshape(2,2,3,4)
expected_slice = values[0,:,4:8]
print('wrong head 1:',wrong[0,1].tolist())
print('source features 4:8:',expected_slice.tolist())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def split_heads(x, heads):
    # TODO: اول محور ویژگی را بشکنید، سپس زمان و Head را جابه‌جا کنید
    return None

In [ ]:
def test_repair():
    result = split_heads(values,2)
    if result is None:
        return False
    assert result.shape==(2,2,3,4)
    assert torch.equal(result[0,1],values[0,:,4:8])
    other = torch.arange(3*7*20).reshape(3,7,20)
    divided = split_heads(other,4)
    assert divided.shape==(3,4,7,5)
    assert divided[2,3,6,4]==other[2,6,19]
    assert torch.equal(divided.transpose(1,2).reshape(3,7,20),other)
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: split_heads')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right">دو قرارداد مستقیماً در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">MiniGPT.forward</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">CausalSelfAttention.forward</code> استفاده می‌شوند. آزمون با اندازه‌های نامساوی کمک می‌کند خطا پشت برابری تصادفی محورهای مختلف پنهان نماند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">کدام آزمونِ مقدار توانست خطایی را ببیند که آزمون <bdi dir="ltr">Shape</bdi> به‌تنهایی از دست می‌داد؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-05/60-bug-clinic.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/60-bug-clinic.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>